# Workshop: Rapid Application Development using Large Language Models
## Part 3: LLM Servers, Tool Use, and Agentic Systems

Practical session (60 min)

This notebook is part of a workshop based on NVIDIA DLI materials
"Rapid Application Development using Large Language Models",
adapted for Google Colab with a T4 GPU.

Credits: Original course materials by NVIDIA Corporation / NVIDIA Deep Learning Institute.

### Learning Objectives

By the end of this notebook you will be able to:

- Connect to a cloud LLM via an OpenAI-compatible API and stream responses
- Understand why this API format is the standard for most LLM providers today
- Ask an LLM to return structured JSON output
- Give an LLM access to tools (function calling) and let it decide when to use them
- Build a simple ReAct agent (Reason + Act) from scratch
- Recreate the same agent using LangGraph with less manual code

In [1]:
!pip install -q openai langchain-groq langgraph langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.1 MB/s eta 0:00:00


### Before you start: get a free Groq API key

1. Go to **[console.groq.com](https://console.groq.com)** and create a free account
2. In the upper right menu, click **API Keys**
3. Click **Create API Key**, give it any name, and click **Submit**
4. Copy the key (it starts with `gsk_`)

**Add the key to Colab Secrets (recommended):**

5. Click the **key icon** in the left Colab sidebar (Secrets)
6. Click **Add new secret**
7. Set Name to `GROQ_API_KEY` and paste your key as the Value
8. Turn on **Notebook access** for this secret
9. The cell below reads it automatically with `userdata.get()`

**Alternative (simpler but less safe):**

Paste the key directly into the `GROQ_API_KEY = ...` line in the cell below.
Do not share your notebook with the key inside it.

The free tier gives you 30 requests per minute and 30,000 tokens per minute,
which is more than enough for this workshop.

In [13]:
from google.colab import userdata
import os

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
print('API key loaded from Colab Secrets.')

os.environ['GROQ_API_KEY'] = GROQ_API_KEY

if GROQ_API_KEY == 'gsk_...':
    print('WARNING: please add your API key using one of the two options above.')

API key loaded from Colab Secrets.


---

## Part 3.1: Connecting to an LLM via API

In Parts 1 and 2, we loaded models directly into the GPU.
This works for small models but has limits:
- Large models (70B+) do not fit in a single GPU
- Loading and unloading takes time
- Multiple users cannot share the same loaded model easily

The solution is an **inference server**: the model runs in one place
and we talk to it over a network using an API.

The most common standard today is the **OpenAI API format**.
Almost every provider uses it: Groq, Ollama, vLLM, NVIDIA NIM, and others.
This means the same Python code works with any of these backends.
Only the `base_url` and `api_key` change.

We use **Groq** because it is free, fast, and requires no local setup.

In [24]:
from openai import OpenAI

MODEL = "openai/gpt-oss-20b"

# Current models on Groq (June 2026):
#   "openai/gpt-oss-20b"    fast, supports tool use, free tier
#   "openai/gpt-oss-120b"   larger, better reasoning, supports tool use
#   "qwen/qwen3.6-27b"      multilingual, supports tool use
#
# Recently deprecated (announced June 17, 2026):
#   "llama-3.1-8b-instant"     -> replaced by openai/gpt-oss-20b
#   "llama-3.3-70b-versatile"  -> replaced by openai/gpt-oss-120b or qwen/qwen3.6-27b
#   "meta-llama/llama-4-scout-17b-16e-instruct" -> replaced by openai/gpt-oss-120b

client = OpenAI(
    base_url = "https://api.groq.com/openai/v1",
    api_key = GROQ_API_KEY,
)

response = client.chat.completions.create(
    model = MODEL,
    messages = [
        {"role": "user", "content": "What is a transformer model? Answer in two sentences."}
    ],
)

print(response.choices[0].message.content)

A transformer model is a deep learning architecture that processes input data using self‑attention mechanisms, allowing it to weigh the importance of each token in relation to all others in a sequence. This design enables highly parallel computation and captures long‑range dependencies, making it especially effective for natural language processing and other sequence‑based tasks.


In [25]:
# Streaming: print tokens as they arrive instead of waiting for the full response

stream = client.chat.completions.create(
    model = MODEL,
    messages = [
        {"role": "user", "content": "Explain the difference between an encoder and a decoder in three sentences."}
    ],
    stream = True,
)

print("Streamed response:")
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()

Streamed response:
An encoder compresses or transforms raw input data (like an image, audio, or text) into a compact, often high‑dimensional representation that captures its essential features. A decoder takes that representation and reconstructs the original data or generates new data from it, essentially performing the inverse operation of the encoder. In many models, such as autoencoders or sequence‑to‑sequence networks, the encoder and decoder work together so the system learns to encode inputs into a useful latent space and then decode that space back into a meaningful output.


In [26]:
# Multi-turn conversation: the model remembers previous messages
# We send the full message history with every request

messages = [
    {"role": "system", "content": "You are a helpful assistant for a machine learning workshop. Keep answers short."}
]

questions = [
    "What is a large language model?",
    "How is it different from a traditional text classifier?",
    "Give one real-world example where you would use each.",
]

for question in questions:
    messages.append({"role": "user", "content": question})
    response = client.chat.completions.create(model=MODEL, messages=messages)
    reply = response.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    print(f"User     : {question}")
    print(f"Assistant: {reply}")
    print()

User     : What is a large language model?
Assistant: A large language model (LLM) is a neural network—usually a transformer—that’s trained on vast amounts of text. It learns statistical patterns of word usage and can predict, generate, or understand language with high accuracy.

User     : How is it different from a traditional text classifier?
Assistant: A traditional text classifier is a fixed‑size model that learns to assign labels (e.g., spam/not spam) to input text.  
An LLM is a huge generative network that learns the joint distribution of language; it can **predict the next word, generate text, translate, answer questions, etc.** In short, classifiers make decisions, LLMs produce and understand text.

User     : Give one real-world example where you would use each.
Assistant: **Large language model** – Chat‑GPT in a customer‑service chatbot that can answer free‑form questions about a product, generate help‑desk articles, or draft emails.  

**Traditional text classifier** – A s

---

## Part 3.2: Structured JSON Output

By default, the model returns plain text.
For many applications we need structured data that we can parse in code.

We can ask the model to return **JSON** by:
1. Adding clear instructions in the prompt
2. Setting `response_format={"type": "json_object"}`

The model will then always return valid JSON.
This is useful for data extraction, classification, and building pipelines.

In [27]:
import json

prompt = (
    "Extract the following fields from the text as JSON:\n"
    "- topic: the main topic (one short phrase)\n"
    "- entities: list of named entities (people, organizations, places)\n"
    "- sentiment: positive, negative, or neutral\n"
    "- key_fact: the single most important fact (one sentence)\n\n"
    "Text: Researchers at MIT announced a new training method that reduces "
    "the cost of fine-tuning large language models by 60%. The team, led by "
    "Dr. Sarah Chen, published their results in Nature yesterday.\n\n"
    "Return only valid JSON. No explanation."
)

response = client.chat.completions.create(
    model = MODEL,
    messages = [{"role": "user", "content": prompt}],
    response_format = {"type": "json_object"},
)

result = json.loads(response.choices[0].message.content)
print(json.dumps(result, indent=2))

{
  "topic": "training method reduces fine-tuning cost",
  "entities": [
    "MIT",
    "Dr. Sarah Chen",
    "Nature"
  ],
  "sentiment": "positive",
  "key_fact": "MIT researchers introduced a training method that cuts fine-tuning cost for large language models by 60%."
}


---

### Exercise 1: Structured Extraction

**Tasks:**
1. Change the text to any news headline or short paragraph
2. Change the fields to extract something different (e.g. product, price, brand)
3. Try requesting a field that is not in the text. What does the model return?

In [28]:
# TODO: change the text and the fields to extract

my_text = (
    "Apple announced at their annual conference that the new iPhone includes "
    "a dedicated AI chip. CEO Tim Cook called it the most advanced mobile processor "
    "ever built. The phone will be available in stores starting next month."
)

my_prompt = (
    "Extract these fields from the text as JSON:\n"
    "- product: product name\n"
    "- company: company name\n"
    "- announcement: what was announced (one sentence)\n"
    "- sentiment: positive, negative, or neutral\n\n"
    f"Text: {my_text}\n\n"
    "Return only valid JSON. No explanation."
)

response = client.chat.completions.create(
    model = MODEL,
    messages = [{"role": "user", "content": my_prompt}],
    response_format = {"type": "json_object"},
)

result = json.loads(response.choices[0].message.content)
print(json.dumps(result, indent=2))

{
  "product": "iPhone",
  "company": "Apple",
  "announcement": "Apple announced at their annual conference that the new iPhone includes a dedicated AI chip.",
  "sentiment": "positive"
}


---

## Part 3.3: Tool Use and Function Calling

A plain LLM can only return text.
**Tool use** gives the model access to external functions it can call.

The model does not run the functions itself.
It decides when and how to call them, we run them, and we send the result back.

The process:
1. We send a question plus a list of available tools (as JSON schema)
2. The model decides if a tool is needed and returns a structured call
3. We execute the tool and send the result back
4. The model uses the result to write its final answer

This pattern is how most AI assistants work today.

In [29]:
import math, json

# Define tools as JSON schemas
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a math expression. Use for any calculation.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A Python math expression, e.g. '2**10' or 'math.sqrt(144)'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "lookup",
            "description": "Look up the definition of a machine learning term.",
            "parameters": {
                "type": "object",
                "properties": {
                    "term": {
                        "type": "string",
                        "description": "The ML term to look up"
                    }
                },
                "required": ["term"]
            }
        }
    }
]

# Implement the tool functions
def calculate(expression):
    try:
        return str(eval(expression, {"math": math, "__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"

GLOSSARY = {
    "transformer" : "A neural network that uses self-attention to process sequences in parallel.",
    "tokenizer"   : "A component that splits text into tokens and maps them to integer IDs.",
    "embedding"   : "A dense vector that represents the meaning of a token or sequence.",
    "attention"   : "A mechanism that lets each token look at and borrow from all other tokens.",
    "fine-tuning" : "Training a pre-trained model further on a smaller task-specific dataset.",
    "llm"         : "Large Language Model: a model with billions of parameters trained on large text corpora.",
    "rag"         : "Retrieval-Augmented Generation: combining a retriever with an LLM for document-based QA.",
}

def lookup(term):
    return GLOSSARY.get(term.lower(), f"Term '{term}' not found in glossary.")

print("Tools defined:", [t["function"]["name"] for t in tools])

Tools defined: ['calculate', 'lookup']


In [32]:
def run_agent(user_message):
    messages = [
        {"role": "system", "content": "You are a helpful ML workshop assistant. Use tools when needed."},
        {"role": "user",   "content": user_message},
    ]

    # First call: model decides if it needs a tool
    response = client.chat.completions.create(
        model = MODEL,
        messages = messages,
        tools = tools,
    )
    msg = response.choices[0].message

    # If the model called a tool, execute it and call again
    if msg.tool_calls:
        messages.append(msg)
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            result = calculate(**args) if name == "calculate" else lookup(**args)
            print(f"  [Tool] {name}({args}) -> {result}")
            messages.append({
                "role"         : "tool",
                "tool_call_id" : tc.id,
                "content"      : result,
            })
        final = client.chat.completions.create(model=MODEL, messages=messages)
        return final.choices[0].message.content

    return msg.content


questions = [
    "What is 2 to the power of 15?",
    "What does 'tokenizer' mean in machine learning?",
    "If a model has 7 billion parameters and each takes 2 bytes in fp16, how many GB is that?",
    "What is RAG and when would you use it?",
]

for q in questions:
    print(f"Question : {q}")
    print(f"Answer   : {run_agent(q)}")
    print()

Question : What is 2 to the power of 15?
  [Tool] calculate({'expression': '2**15'}) -> 32768
Answer   : \(2^{15}=32,768\)

Question : What does 'tokenizer' mean in machine learning?
Answer   : ### In a nutshell

A **tokenizer** is a piece of software (or a small algorithm) that takes raw input—most often text—and breaks it up into a sequence of *tokens*.  
Tokens are the atomic units that a machine‑learning model can work with, such as words, sub‑words, characters, or even numeric/visual units in other modalities. Once the data is tokenized, each token is usually mapped to a unique integer ID that can be fed into a neural network.

---

## Why tokenization matters

| Problem | Why you need a tokenizer | What it does |
|---------|--------------------------|--------------|
| **Variable input lengths** | Models expect a fixed‑size sequence. | Breaks a sentence into a list of tokens that can be padded or truncated. |
| **Vocabulary size** | Too many words → huge embedding matrix. | Splits

---

### Exercise 2: Add a New Tool

**Tasks:**
1. The `convert_units` tool is already defined below
2. Ask questions that need this tool
3. Add your own entry to the GLOSSARY dictionary and test it

In [33]:
new_tool = {
    "type": "function",
    "function": {
        "name": "convert_units",
        "description": "Convert between common units. Supported: mb_to_gb, celsius_to_fahrenheit.",
        "parameters": {
            "type": "object",
            "properties": {
                "value":      {"type": "number", "description": "The value to convert"},
                "conversion": {"type": "string", "description": "mb_to_gb or celsius_to_fahrenheit"}
            },
            "required": ["value", "conversion"]
        }
    }
}

def convert_units(value, conversion):
    if conversion == "mb_to_gb":
        return f"{value} MB = {value / 1024:.3f} GB"
    elif conversion == "celsius_to_fahrenheit":
        return f"{value} C = {value * 9/5 + 32:.1f} F"
    return f"Unknown conversion: {conversion}"

my_tools = tools + [new_tool]

def run_agent_v2(user_message):
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use tools when needed."},
        {"role": "user",   "content": user_message},
    ]
    response = client.chat.completions.create(model=MODEL, messages=messages, tools=my_tools)
    msg = response.choices[0].message
    if msg.tool_calls:
        messages.append(msg)
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            if name == "calculate":      result = calculate(**args)
            elif name == "lookup":        result = lookup(**args)
            elif name == "convert_units": result = convert_units(**args)
            else: result = "Unknown tool"
            print(f"  [Tool] {name}({args}) -> {result}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
        final = client.chat.completions.create(model=MODEL, messages=messages)
        return final.choices[0].message.content
    return msg.content

# TODO: try questions that use the new tool, or add your own
test_questions = [
    "How many GB is 4096 MB?",
    "What is 37 degrees Celsius in Fahrenheit?",
    # TODO: add a word to GLOSSARY and test it here
]
for q in test_questions:
    print(f"Question : {q}")
    print(f"Answer   : {run_agent_v2(q)}")
    print()

Question : How many GB is 4096 MB?
  [Tool] convert_units({'conversion': 'mb_to_gb', 'value': 4096}) -> 4096 MB = 4.000 GB
Answer   : 4096 MB is exactly **4 GB** (using the 1 GB = 1024 MB conversion).

Question : What is 37 degrees Celsius in Fahrenheit?
  [Tool] convert_units({'conversion': 'celsius_to_fahrenheit', 'value': 37}) -> 37 C = 98.6 F
Answer   : 37 °C is **98.6 °F**.



---

## Part 3.4: The ReAct Loop

**ReAct** stands for Reason + Act.
It is a general pattern for building agents:

1. The model reads the question and writes its reasoning (Thought)
2. It decides on an action (call a tool)
3. The tool result is added to the conversation (Observation)
4. The model reasons again with the new information
5. This loop repeats until the model writes a final Answer

Unlike the function calling in Part 3.3 which uses grammar enforcement,
ReAct uses plain text formatting.
The model writes its thoughts before acting, making the reasoning visible and easy to debug.

In [37]:
def react_agent(question, max_steps=5):
    system = (
        "You are a helpful assistant. Do NOT use function calling or API tools.\n"
        "Instead, write everything as plain text in this exact format:\n\n"
        "Thought: [your reasoning about what to do]\n"
        "Action: calculate OR lookup\n"
        "Input: [the expression or term]\n\n"
        "When you have the final answer, write:\n"
        "Thought: [your final reasoning]\n"
        "Answer: [your complete answer]\n\n"
        "For math, write Action: calculate and Input: the expression.\n"
        "For definitions, write Action: lookup and Input: the term.\n"
        "Always include a Thought line. Never use JSON or function call syntax."
    )

    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": question},
    ]

    print(f"Question: {question}\n")

    for step in range(max_steps):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            # no tools parameter here: plain text output only
        )
        text = response.choices[0].message.content
        if text is None:
            text = ""
        text = text.strip()
        messages.append({"role": "assistant", "content": text})

        print(f"--- Step {step + 1} ---")
        print(text)
        print()

        if "Answer:" in text:
            break

        if "Action:" in text and "Input:" in text:
            lines = text.split("\n")
            action = next((l.split("Action:")[-1].strip() for l in lines if "Action:" in l), "")
            inp    = next((l.split("Input:")[-1].strip()  for l in lines if "Input:"  in l), "")

            if "calculate" in action.lower():  obs = calculate(inp)
            elif "lookup"  in action.lower():  obs = lookup(inp)
            else:                               obs = "Unknown action. Use calculate or lookup."

            observation = f"Observation: {obs}"
            print(observation)
            print()
            messages.append({"role": "user", "content": observation})


react_agent("What is the square root of 2025, and what does fine-tuning mean?")

Question: What is the square root of 2025, and what does fine-tuning mean?

--- Step 1 ---
Thought: Calculate the square root of 2025.  
Action: calculate  
Input: sqrt(2025)

Thought: Look up the definition of fine-tuning.  
Action: lookup  
Input: fine-tuning

Thought: Combine the computed value and the definition into a concise answer.  
Answer: The square root of 2025 is **45**. Fine‑tuning is the process of making small, targeted adjustments to a pre‑trained model (or any system) so that it performs better on a specific task or dataset.



---

## Part 3.5: The Same Agent with LangGraph

The ReAct loop in Part 3.4 works but requires manual code for:
- Parsing the model output to find tool calls
- Routing to the right tool function
- Managing the message history
- Deciding when to stop the loop

**LangGraph** handles all of this for you.
You define the agent as a graph with nodes and edges:
- A **node** is one processing step (call the LLM, or execute a tool)
- An **edge** is a transition between steps

LangGraph manages state, routing, and the loop automatically.
The result is the same behavior as Part 3.4 but with much less code.

In [38]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.graph import StateGraph, MessagesState, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import HumanMessage

# Define tools using the LangChain @tool decorator
@tool
def lg_calculate(expression: str) -> str:
    """Evaluate a mathematical expression."""
    return calculate(expression)

@tool
def lg_lookup(term: str) -> str:
    """Look up the definition of a machine learning term."""
    return lookup(term)

lg_tools = [lg_calculate, lg_lookup]

# Create the LLM with tools attached
llm = ChatGroq(model=MODEL, api_key=GROQ_API_KEY)
llm_with_tools = llm.bind_tools(lg_tools)

# Agent node: calls the LLM
def agent_node(state: MessagesState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# Build the graph
builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(lg_tools))
builder.set_entry_point("agent")
builder.add_conditional_edges("agent", tools_condition)  # route to tools or END
builder.add_edge("tools", "agent")                        # after tools, go back to agent
graph = builder.compile()

print("LangGraph agent ready.")

LangGraph agent ready.


In [39]:
# Run the same question as in Part 3.4 and compare the output
question = "What is the square root of 2025, and what does fine-tuning mean?"

result = graph.invoke({"messages": [HumanMessage(content=question)]})

print(f"Question: {question}\n")
print(f"Answer  : {result['messages'][-1].content}")
print()
print(f"Total messages in graph: {len(result['messages'])}")
print("(human -> agent thought -> tool call -> tool result -> final answer)")

Question: What is the square root of 2025, and what does fine-tuning mean?

Answer  : - **Square root of 2025**  
  \[
  \sqrt{2025}=45
  \]
  (Because \(45 \times 45 = 2025\).)

- **Fine‑tuning (in machine learning)**  
  Fine‑tuning is the process of taking a model that has already been trained on a large, general dataset (often called a “pre‑trained” model) and then continuing to train it on a smaller, task‑specific dataset. The goal is to adapt the model’s parameters to better fit the new domain or problem while preserving the knowledge it learned during the initial training.  

  Key points:
  1. **Transfer learning**: The pre‑trained model provides a strong starting point, reducing the amount of data and time needed for the new task.
  2. **Parameter updates**: Typically only a subset of the model’s layers or a small learning rate is used to avoid catastrophic forgetting of the original knowledge.
  3. **Applications**: Common in natural language processing (e.g., fine‑tuning BER

---

## Mini Project: Content Generation Pipeline

A pipeline chains multiple steps where the output of one step
becomes the input of the next.

Here we build a simple content pipeline:
1. Start with an image description (as if it came from a vision model like ViT+GPT2 from Part 2)
2. Use structured JSON output to generate a tweet and a caption for each image
3. Show all results

This connects the structured output technique from Part 3.2
with the multi-step pipeline concept used in real AI applications.

In [40]:
# Image descriptions (in a real pipeline these would come from ViT+GPT2 or a VLM)
image_descriptions = [
    "A happy golden Labrador sitting on green grass, looking at the camera with a friendly expression.",
    "A fluffy tabby cat sitting calmly on a white surface, alert and well-groomed.",
    "The Eiffel Tower photographed from the ground during daytime against a clear blue sky.",
]

def generate_content(description):
    prompt = (
        "Given this image description, generate two pieces of content as JSON:\n"
        "- tweet: a short engaging tweet (max 280 characters, include hashtags)\n"
        "- caption: a short Instagram-style caption (1-2 sentences, friendly tone)\n\n"
        f"Image: {description}\n\n"
        "Return only valid JSON with keys tweet and caption."
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)


print("Content Generation Pipeline\n")
print("=" * 60)

for i, desc in enumerate(image_descriptions):
    content = generate_content(desc)
    print(f"Image {i+1}: {desc[:60]}...")
    print(f"Tweet  : {content.get('tweet', 'N/A')}")
    print(f"Caption: {content.get('caption', 'N/A')}")
    print()

Content Generation Pipeline

Image 1: A happy golden Labrador sitting on green grass, looking at t...
Tweet  : Meet our sunshine in fur! 🌞🐾 This golden Labrador is ready to steal your heart from the green grass. Come wag with us! #GoldenRetriever #DogLove #PetsofInstagram #FurryFriend
Caption: Sunshine and smiles on a sunny day! This golden Labrador can’t wait to meet you. 🌞🐶

Image 2: A fluffy tabby cat sitting calmly on a white surface, alert ...
Tweet  : Just spotted the ultimate zen cat 🐾✨ This fluffy tabby is mastering the art of calm on a white backdrop. Pure paw-sitivity! #CatLife #FelineZen #PetsofInstagram
Caption: Meet our fluffy tabby friend, lounging on a crisp white background with impeccable style! Pure serenity and cuteness in every purr.

Image 3: The Eiffel Tower photographed from the ground during daytime...
Tweet  : Standing under the iconic Eiffel Tower today, feeling the Parisian breeze in the bright blue sky. #EiffelTower #Paris #Travel #Wanderlust #Iconic
Caption

---

## Wrapping Up

In this notebook you have:

- Connected to a cloud LLM using the OpenAI-compatible API format
- Streamed responses and built a multi-turn conversation
- Extracted structured JSON output from an LLM
- Defined tools and let the model decide when to call them
- Built a ReAct agent manually with visible reasoning steps
- Recreated the same agent in LangGraph with less code
- Built a simple content generation pipeline

**Key points:**

- The OpenAI API format is the standard today. Groq, Ollama, vLLM, and NVIDIA NIM all use it.
  Switching between providers changes only base_url and api_key.
- Structured JSON output lets you connect LLM responses to any downstream code.
- Tool use is the foundation of modern AI assistants. The model reasons; we execute.
- LangGraph removes the need to write the agent loop, state management, and routing by hand.

**Where to go next:**
- Add more tools to your LangGraph agent (web search, database lookup, file reading)
- Try a larger Groq model (llama-3.3-70b-versatile) for better reasoning
- Explore LangGraph checkpointing to give your agent persistent memory across sessions